In [ ]:

import sys, json, hashlib
from pathlib import Path

project_root = Path(globals().get("PROJECT_ROOT", Path.cwd()))
_original_sys_path = sys.path[:]
sys.path.insert(0, str(project_root))
import benchmark
from free_threading_core import reference_image, image_digest
sys.path[:] = _original_sys_path

width, height, iterations = 32, 24, 40
workers = min(2, benchmark.effective_cpus()["effective_cpus"])

src = (project_root / "source" / "original.ipynb").read_bytes()
source_sha256 = hashlib.sha256(src).hexdigest()
assert source_sha256 == "ea34493fc5caaa6062dfa9b4210b1d0620f205cce4cc117c9293155f75a1c911"
eng = (project_root / "agilab_pool.py").read_bytes()
engine_sha256 = hashlib.sha256(eng).hexdigest()
assert engine_sha256 == "305ba174348e92376b08be149b488fc41983de04c5b2564695493769fc21f066"
print(f"Stage1: files verified, workers={workers}")



In [ ]:

counts = reference_image(width, height, iterations)
assert len(counts) == width * height
assert all(isinstance(c, int) and 0 <= c <= iterations for c in counts)
digest = image_digest(counts)
print(f"Stage2: {len(counts)} counts, digest={digest[:16]}...")



In [ ]:

report = benchmark.run_benchmark(width=width, height=height, iterations=iterations, workers=workers, repeats=1)
assert len(report["runs"]) == 6
assert len(report["summary"]) == 6
assert report["same_work_verified"] is True
assert report["digest"] == digest
for run in report["runs"]:
    ordered = sorted(run["records"], key=lambda r: r["row_start"])
    concat = [c for rec in ordered for c in rec["counts"]]
    assert concat == counts
    assert run["digest"] == digest
for s in report["summary"]:
    print(f"{s['mode']}/{s['role']}: wall={s['wall_seconds']:.4f} engine={s['engine_seconds']:.4f} speedup={s['speedup']:.3f}")



In [ ]:

modes = sorted(set(r["mode"] for r in report["runs"]))
artifact = {"results": {"width": width, "height": height, "iterations": iterations, "digest": digest, "same_work_verified": report["same_work_verified"], "modes": modes, "case_count": len(report["runs"]), "source_sha256": source_sha256, "engine_sha256": engine_sha256}}
with open(Path.cwd() / "results.json", "w") as f:
    json.dump(artifact, f, allow_nan=False)
print(f"Stage4: verified, {len(modes)} modes, {len(report['runs'])} cases, digest={digest[:16]}...")